# Task 2: Structured Channel Pruning (Regression-Based)

Implementation of Task 2(a): Regression-Based Channel Pruning using the method from He et al., 2017.

## ⚠️ Implementation Update
**FIXED (Dec 2024)**: Now correctly implements the He et al. 2017 method with:
- ✅ LASSO regression for channel selection (Step 2)
- ✅ Weight reconstruction via least squares (Step 3)
- ✅ No intermediate training (only mathematical computation)

**Key Differences from Task 1:**
- **Unstructured Pruning (Task 1)**: Prunes individual weights → irregular sparsity, needs sparse ops
- **Structured Pruning (Task 2)**: Prunes entire channels → smaller dense networks, real speedup

**Method Overview:**
1. Use LASSO regression to select important channels
2. Reconstruct weights via least squares (mathematical, NO training)
3. Fine-tune ONLY after ALL layers pruned

**Paper:** He et al., 2017 - "Channel pruning for accelerating very deep neural networks"

**GitHub:** https://github.dev/ethanhe42/channel-pruning

## Part 1: Setup & Configuration

In [ ]:
!pip install torchprofile
# Cell 1: Import Libraries
import os
import io
import time
import copy
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
from torch.profiler import profile, ProfilerActivity
from torchprofile import profile_macs

# For LASSO regression
from sklearn.linear_model import Lasso
import warnings
warnings.filterwarnings('ignore')  # Suppress sklearn convergence warnings

print("✓ All libraries imported successfully")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Configuration
# BASE_PATH: Change this for local vs Colab
# Local: BASE_PATH = './'
BASE_PATH = '/content/drive/MyDrive/AI624-Edge-Devices/pa1.1'

DEVICE = 'cuda'
BATCHSIZE = 128

# We use 1280 (10 batches) for stable LASSO regression
CALIBRATION_SAMPLES = 512

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Base Path: {BASE_PATH}")
print(f"Device: {DEVICE.upper()}")
print(f"Batch Size: {BATCHSIZE}")
print(f"Calibration Samples: {CALIBRATION_SAMPLES}")
print("="*80)

## Part 2: Helper Functions (from task1b_final.ipynb)

In [ ]:
#
# Evaluate model accuracy (Top-1 and Top-5)
#
def evaluate_accuracy(model, data_loader):
    """Evaluate model accuracy with Top-1 and Top-5 metrics."""
    model.eval()

    correct_top1 = 0
    correct_top5 = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            # Top-1 accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct_top1 += (predicted == labels).sum().item()

            # Top-5 accuracy
            _, top5_pred = outputs.topk(5, 1, True, True)
            top5_pred = top5_pred.t()
            correct_top5 += top5_pred.eq(labels.view(1, -1).expand_as(top5_pred)).sum().item()

    top1_accuracy = 100 * correct_top1 / total
    top5_accuracy = 100 * correct_top5 / total

    return top1_accuracy, top5_accuracy

In [ ]:
#
# Profile model memory and latency
#
def profile_model(model, dataloader):
    """Profile model for memory and latency metrics."""
    # Get one batch for profiling
    inputs, _ = next(iter(dataloader))
    inputs = inputs.to(DEVICE)

    # Measure latency outside of profiler for more accurate timing
    latencies_ms = []
    with torch.no_grad():
        for _ in range(10):  # Run 10 iterations for average
            start_time = time.time()
            output = model(inputs)
            end_time = time.time()
            latencies_ms.append((end_time - start_time) * 1000)

    # Calculate average latency
    avg_latency_ms = sum(latencies_ms) / len(latencies_ms)

    # Profile memory usage
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA] if DEVICE == 'cuda' else [ProfilerActivity.CPU],
        record_shapes=True,
        profile_memory=True,
        with_stack=False
    ) as prof:
        with torch.no_grad():
            _ = model(inputs)

    # Extract memory metrics
    if DEVICE == 'cuda':
        peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
        torch.cuda.reset_peak_memory_stats()
    else:
        peak_memory_mb = 0.0

    return {
        'latency_ms': avg_latency_ms,
        'peak_memory_mb': peak_memory_mb
    }

In [ ]:
#
# Get model size in MB
#
def get_model_size_mb(model):
    """Get model size in MB by serializing it."""
    buffer = io.BytesIO()  # Create in-memory buffer
    torch.save(model.state_dict(), buffer)  # Serialize model
    size_bytes = buffer.tell()  # Get bytes written
    size_mb = size_bytes / (1024 * 1024)  # Convert to MB
    return size_mb

In [ ]:
#
# Count MACs (Multiply-Accumulate operations)
#
def count_model_macs(model, input_size=(1, 3, 32, 32)):
    """Count MACs for model."""
    input_tensor = torch.randn(input_size).to(DEVICE)
    return profile_macs(model, input_tensor)

## Part 3: Load Models & Datasets

In [ ]:
#
# Load pretrained VGG16-BN model
#
def load_pretrained_model(dataset_name='cifar10'):
    """Load pretrained VGG16-BN model from local files."""
    model_path = os.path.join(BASE_PATH, 'models', f'{dataset_name}_vgg16.pt')

    # Clear CUDA cache if using CUDA
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # Load state dict to CPU first to avoid CUDA errors
    state_dict = torch.load(model_path, map_location='cpu')

    # Determine classifier architecture from state dict
    first_classifier_weight_shape = state_dict['classifier.0.weight'].shape
    classifier_input_features = first_classifier_weight_shape[1]  # Input features

    # Create VGG16-BN with standard architecture
    model = models.vgg16_bn(pretrained=False)

    # Adapt for CIFAR (32×32 images)
    # Change avgpool from (7,7) to (1,1) for CIFAR
    model.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    # Adapt classifier for CIFAR
    num_classes = 10 if dataset_name == 'cifar10' else 100

    # Rebuild classifier to match saved model architecture
    model.classifier = nn.Sequential(
        nn.Linear(classifier_input_features, 512),
        nn.ReLU(True),
        nn.Dropout(p=0.5),
        nn.Linear(512, 512),
        nn.ReLU(True),
        nn.Dropout(p=0.5),
        nn.Linear(512, num_classes)
    )

    # Load the state dict
    model.load_state_dict(state_dict)

    # Now move to device
    model.to(DEVICE)

    print(f"Loaded pretrained model from {model_path}")
    print(f"  Classifier input features: {classifier_input_features}")
    print(f"  Number of classes: {num_classes}")

    return model

In [ ]:
#
# Load CIFAR dataset from local files
#
def load_dataset(dataset_name='cifar10'):
    """Load CIFAR dataset from local files."""
    # Normalization values for CIFAR
    normalize = T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                           std=[0.2023, 0.1994, 0.2010])

    transform = T.Compose([T.ToTensor(), normalize])

    if dataset_name == 'cifar10':
        data_path = os.path.join(BASE_PATH, 'datasets')
        train_dataset = torchvision.datasets.CIFAR10(
            root=data_path, train=True, transform=transform, download=False
        )
        test_dataset = torchvision.datasets.CIFAR10(
            root=data_path, train=False, transform=transform, download=False
        )
    else:  # cifar100
        data_path = os.path.join(BASE_PATH, 'datasets')
        train_dataset = torchvision.datasets.CIFAR100(
            root=data_path, train=True, transform=transform, download=False
        )
        test_dataset = torchvision.datasets.CIFAR100(
            root=data_path, train=False, transform=transform, download=False
        )

    return train_dataset, test_dataset

## Part 4: Layer-wise Sparsity Targets (from Task 1a)

**Important:** For Task 2, we skip the ENTIRE first convolutional block and start from features.7 (second block).

In [ ]:
#
# Layer-wise sparsity targets for structured channel pruning
# Based on sensitivity analysis from Task 1a
#
# NOTE: features.0 and features.3 have 0.0 sparsity (skip first block entirely)
# Linear layers NOT pruned in structured pruning
#
cifar10_layer_sparsity = {
    # Block 1 - SKIP ENTIRELY per requirement
    'features.0': 0.0,     # Conv1_1 - Skip
    'features.3': 0.0,     # Conv1_2 - Skip

    # Block 2 - START PRUNING HERE
    'features.7': 0.70,    # Conv2_1 - Moderate sensitivity
    'features.10': 0.60,   # Conv2_2 - More sensitive

    # Block 3
    'features.14': 0.60,   # Conv3_1 - More sensitive
    'features.17': 0.70,   # Conv3_2 - Robust
    'features.20': 0.70,   # Conv3_3 - Robust

    # Block 4
    'features.24': 0.80,   # Conv4_1 - Very robust
    'features.27': 0.90,   # Conv4_2 - Super robust
    'features.30': 0.90,   # Conv4_3 - Super robust

    # Block 5
    'features.34': 0.90,   # Conv5_1 - Super robust
    'features.37': 0.90,   # Conv5_2 - Super robust
    'features.40': 0.90,   # Conv5_3 - Super robust
}

cifar100_layer_sparsity = {
    # Block 1 (skip)
    'features.0': 0.0,     # Skip - first block
    'features.3': 0.0,     # Skip - first block

    # Block 2 (START HERE)
    'features.7': 0.60,    # Sensitive
    'features.10': 0.50,   # Very sensitive

    # Block 3
    'features.14': 0.50,   # Very sensitive
    'features.17': 0.60,   # Moderate
    'features.20': 0.60,   # Moderate

    # Block 4
    'features.24': 0.70,   # Robust
    'features.27': 0.85,   # Robust
    'features.30': 0.80,   # Robust

    # Block 5
    'features.34': 0.85,   # Very robust
    'features.37': 0.90,   # Super robust
    'features.40': 0.90,   # Super robust
}

## Part 5: Core Channel Pruning Algorithm

### Step 1: Collect Input-Output Pairs

In [ ]:
#
# Collect input-output pairs for a specific convolutional layer
#
def collect_layer_io_pairs(model, layer_name, calibration_loader):
    """
    Collect input-output pairs for a specific convolutional layer.

    Mathematical Operations:
    1. Forward pass through network up to target layer
    2. Extract input feature maps X ∈ R^(N×c×h×w)
    3. Unfold to get sliding windows: X_unf ∈ R^((N·L)×(c·kh·kw))
    4. Reshape weights: W_mat ∈ R^((c·kh·kw)×n)
    5. Compute output: Y = X_unf @ W_mat ∈ R^((N·L)×n)

    Args:
        model: VGG16-BN model
        layer_name: Name of target conv layer (e.g., 'features.7')
        calibration_loader: DataLoader with calibration samples

    Returns:
        X_unf: Unfolded input features
        Y: Output activations
        W_mat: Reshaped weight matrix
        conv_params: Dictionary with layer parameters
    """
    # Get the conv layer module
    conv_layer = None
    for name, module in model.named_modules():
        if name == layer_name and isinstance(module, nn.Conv2d):
            conv_layer = module
            break

    if conv_layer is None:
        raise ValueError(f"Layer {layer_name} not found or not a Conv2d layer")

    # Extract unfold parameters from the original conv layer
    kernel_size = conv_layer.kernel_size
    stride = conv_layer.stride
    padding = conv_layer.padding
    dilation = conv_layer.dilation

    # Collect input feature maps using hook
    inputs = []
    def hook_fn(module, input, output):
        inputs.append(input[0].detach().cpu())

    hook = conv_layer.register_forward_hook(hook_fn)

    # Forward pass on calibration data
    model.eval()
    with torch.no_grad():
        for images, _ in calibration_loader:
            images = images.to(DEVICE)
            _ = model(images)

    hook.remove()

    # Concatenate all inputs: X ∈ R^(N×c×h×w)
    X = torch.cat(inputs, dim=0)

    # Apply unfold with EXACT same parameters as conv layer
    X_unfolded = F.unfold(
        X,
        kernel_size=kernel_size,
        stride=stride,
        padding=padding,
        dilation=dilation
    )  # Shape: (N, c*kh*kw, L)

    # Reshape to (N*L, c*kh*kw) as required
    N, c_kh_kw, L = X_unfolded.shape
    X_unf = X_unfolded.permute(0, 2, 1).reshape(N * L, c_kh_kw)

    # Get weights and reshape: W ∈ R^(n×c×kh×kw) → W_mat ∈ R^(c*kh*kw, n)
    W = conv_layer.weight.data.cpu()  # (n, c, kh, kw)
    n, c, kh, kw = W.shape
    W_mat = W.view(n, c * kh * kw).t()  # (c*kh*kw, n)

    # Compute output: Y = X_unf @ W_mat
    Y = X_unf @ W_mat  # (N*L, n)

    return X_unf, Y, W_mat, {
        'kernel_size': kernel_size,
        'stride': stride,
        'padding': padding,
        'dilation': dilation,
        'in_channels': c,
        'out_channels': n,
        'kh': kh,
        'kw': kw
    }

### Step 2: LASSO Channel Selection

In [ ]:
#
# Binary search to find λ that gives exactly target_channels
#
def binary_search_lambda(Z, Y, target_channels, max_iter=50):
    """
    Binary search to find λ that gives exactly target_channels.

    Strategy:
    1. Find λ_max: smallest λ that gives all zeros
    2. Set λ_min = 0 (gives all channels)
    3. Binary search between λ_min and λ_max

    Args:
        Z: Design matrix (N*L*n, c) - each column is one Zi
        Y: Target output (N*L*n, 1) - flattened
        target_channels: Desired number of non-zero channels

    Returns:
        selected_channels: Indices of selected channels
        beta_values: Non-zero β coefficients
    """
    # Find λ_max (all coefficients become zero)
    lambda_max = np.abs(Z.T @ Y).max() / Z.shape[0]
    lambda_min = 0

    best_channels = None
    best_beta = None

    for iteration in range(max_iter):
        lambda_val = (lambda_min + lambda_max) / 2

        # Solve LASSO
        lasso = Lasso(alpha=lambda_val, max_iter=10000, tol=1e-4)
        lasso.fit(Z, Y.ravel())
        beta = lasso.coef_

        # Count non-zero coefficients
        non_zero_count = np.sum(np.abs(beta) > 1e-10)

        if non_zero_count == target_channels:
            # Perfect match!
            selected_channels = np.where(np.abs(beta) > 1e-10)[0]
            return selected_channels, beta[selected_channels]

        elif non_zero_count > target_channels:
            # Too many channels, increase λ
            lambda_min = lambda_val
            best_channels = np.where(np.abs(beta) > 1e-10)[0]
            best_beta = beta[best_channels]
            # If close, select top-k by magnitude
            if non_zero_count - target_channels <= 5:
                non_zero_idx = np.where(np.abs(beta) > 1e-10)[0]
                beta_magnitudes = np.abs(beta[non_zero_idx])
                # Keep top target_channels
                top_k_indices = np.argsort(beta_magnitudes)[-target_channels:]
                selected_channels = non_zero_idx[top_k_indices]
                return selected_channels, beta[selected_channels]
        else:
            # Too few channels, decrease λ
            lambda_max = lambda_val
            if best_channels is None or non_zero_count > len(best_channels):
                best_channels = np.where(np.abs(beta) > 1e-10)[0]
                best_beta = beta[best_channels]

    # If didn't converge to exact number, return closest result
    if best_channels is not None and len(best_channels) >= target_channels:
        # Select top target_channels by magnitude
        beta_magnitudes = np.abs(best_beta)
        top_k_indices = np.argsort(beta_magnitudes)[-target_channels:]
        return best_channels[top_k_indices], best_beta[top_k_indices]

    return best_channels, best_beta

In [ ]:
#
# Solve LASSO regression for channel selection
#
def solve_channel_selection_lasso(X_unf, Y, W_mat, target_channels, conv_params):
    """
    Solve LASSO regression for channel selection.

    Mathematical Formulation:
    β̂ = argmin_β { (1/2N) ||Y - Σ(βi·Zi)||²_F + λ||β||₁ }

    where:
    - Zi = Xi @ Wi^T for each channel i
    - Xi is slice of X_unf for channel i (rows with that channel's features)
    - Wi is slice of W for channel i

    Args:
        X_unf: Unfolded input ∈ R^((N·L)×(c·kh·kw))
        Y: Target output ∈ R^((N·L)×n)
        W_mat: Weight matrix ∈ R^((c·kh·kw)×n)
        target_channels: Number of channels to keep
        conv_params: Conv layer parameters

    Returns:
        selected_channels: Indices of channels to keep
        beta_values: β coefficients for selected channels
    """
    N_L, c_kh_kw = X_unf.shape
    n = Y.shape[1]
    c = conv_params['in_channels']
    kh = conv_params['kh']
    kw = conv_params['kw']

    # Step 1: Construct design matrix by computing Zi for each channel
    Z_list = []
    for i in range(c):
        # Extract Xi: features for channel i
        start_idx = i * kh * kw
        end_idx = (i + 1) * kh * kw
        Xi = X_unf[:, start_idx:end_idx]  # (N*L, kh*kw)

        # Extract Wi: weights for channel i
        Wi = W_mat[start_idx:end_idx, :]  # (kh*kw, n)

        # Compute Zi = Xi @ Wi
        Zi = Xi @ Wi  # (N*L, n)
        Z_list.append(Zi)

    # Stack all Zi HORIZONTALLY to create design matrix
    # Each Zi becomes a "feature" in the LASSO problem
    # Design matrix shape: (N*L*n, c) after flattening
    Z = torch.stack(Z_list, dim=0)  # (c, N*L, n)
    Z = Z.permute(1, 2, 0).reshape(-1, c)  # (N*L*n, c)
    Y_flat = Y.reshape(-1, 1)  # (N*L*n, 1)

    # Convert to numpy for sklearn
    Z_np = Z.numpy()
    Y_np = Y_flat.numpy()

    # Step 2: Binary search for λ that gives exactly target_channels
    selected_channels, beta = binary_search_lambda(
        Z_np, Y_np, target_channels
    )

    return selected_channels, beta

### Step 3: Weight Reconstruction (Mathematical, NOT Training)

In [ ]:
#
# Reconstruct optimal weights for selected channels via least squares
#
def reconstruct_weights(X_unf, Y, selected_channels, beta_values, conv_params):
    """
    Reconstruct optimal weights for selected channels via least squares.

    Mathematical Formulation:
    W' = argmin_W ||Y - X' @ W'^T||²_F

    where X' = [β₁X₁, β₂X₂, ..., βc₀Xc₀] (reduced design matrix)

    Solution via least squares (closed-form, no iteration):
    W' = (X'^T @ X')^(-1) @ X'^T @ Y

    This is the key innovation of He et al.: maintaining network functionality
    through mathematical optimization, not iterative training.

    Args:
        X_unf: Original unfolded input ∈ R^((N·L)×(c·kh·kw))
        Y: Target output ∈ R^((N·L)×n)
        selected_channels: Indices of kept channels
        beta_values: β coefficients for scaling
        conv_params: Original conv layer parameters

    Returns:
        W_new: Reconstructed weights ∈ R^(n×c₀×kh×kw)
    """
    kh = conv_params['kh']
    kw = conv_params['kw']
    n = conv_params['out_channels']
    c0 = len(selected_channels)  # Number of kept channels

    # Step 1: Construct reduced design matrix X'
    X_prime_list = []
    for idx, channel_idx in enumerate(selected_channels):
        # Extract features for this channel
        start_idx = channel_idx * kh * kw
        end_idx = (channel_idx + 1) * kh * kw
        Xi = X_unf[:, start_idx:end_idx]  # (N*L, kh*kw)

        # Scale by β coefficient
        Xi_scaled = beta_values[idx] * Xi
        X_prime_list.append(Xi_scaled)

    # Concatenate to form reduced design matrix
    X_prime = torch.cat(X_prime_list, dim=1)  # (N*L, c0*kh*kw)

    # Step 2: Solve least squares problem
    # W' = argmin ||Y - X'W'^T||²_F
    # Solution: W'^T = (X'^T X')^(-1) X'^T Y
    W_prime_T = torch.linalg.lstsq(X_prime, Y).solution  # (c0*kh*kw, n)

    # Step 3: Reshape back to conv weight format
    W_prime = W_prime_T.t()  # (n, c0*kh*kw)
    W_new = W_prime.view(n, c0, kh, kw)  # (n, c0, kh, kw)

    return W_new

## Part 6: Architecture Update Functions

In [ ]:
#
# Update layer with pruned channels
#
def update_layer_architecture(model, layer_name, selected_out_channels, W_new, selected_in_channels=None):
    """
    Update a convolutional layer with pruned channels.

    Args:
        model: VGG16-BN model
        layer_name: Name of layer to update (e.g., 'features.7')
        selected_out_channels: Indices of output channels to keep
        W_new: New weight tensor (n_out, c_in, kh, kw)
        selected_in_channels: Indices of input channels (if previous layer was pruned)
    """
    # Parse layer name
    parts = layer_name.split('.')
    module_name = parts[0]  # 'features'
    layer_idx = int(parts[1])  # e.g., 7

    # Get the sequential module
    sequential_module = getattr(model, module_name)
    old_conv = sequential_module[layer_idx]

    # Create new conv layer with updated channels
    new_conv = nn.Conv2d(
        in_channels=W_new.shape[1],
        out_channels=W_new.shape[0],
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        dilation=old_conv.dilation,
        groups=1,
        bias=old_conv.bias is not None
    )

    # Copy weights
    new_conv.weight.data = W_new.to(DEVICE)

    # Copy bias if exists
    if old_conv.bias is not None:
        new_conv.bias.data = old_conv.bias.data[selected_out_channels].to(DEVICE)

    # Replace the layer
    sequential_module[layer_idx] = new_conv

    # Update BatchNorm layer (next layer after conv)
    if layer_idx + 1 < len(sequential_module):
        next_layer = sequential_module[layer_idx + 1]
        if isinstance(next_layer, nn.BatchNorm2d):
            old_bn = next_layer
            new_bn = nn.BatchNorm2d(W_new.shape[0])

            # Copy statistics for kept channels
            new_bn.weight.data = old_bn.weight.data[selected_out_channels].to(DEVICE)
            new_bn.bias.data = old_bn.bias.data[selected_out_channels].to(DEVICE)
            new_bn.running_mean = old_bn.running_mean[selected_out_channels].to(DEVICE)
            new_bn.running_var = old_bn.running_var[selected_out_channels].to(DEVICE)

            sequential_module[layer_idx + 1] = new_bn

In [ ]:
#
# Update classifier input dimension after pruning last conv layer
#
def update_classifier_input_dimension(model, last_conv_channels):
    """
    Update the first linear layer's input dimension after pruning last conv block.

    When the last conv layer (features.40) is pruned, the flattened feature size changes.

    Args:
        model: VGG16-BN model
        last_conv_channels: Number of output channels in last conv layer after pruning
    """
    # Calculate new flattened size (for CIFAR: 1x1 spatial after adaptive pooling)
    new_input_features = last_conv_channels * 1 * 1  # VGG uses AdaptiveAvgPool2d((1,1))

    # Get current first linear layer
    first_linear = model.classifier[0]
    out_features = first_linear.out_features

    # Create new linear layer with correct input dimension
    new_first_linear = nn.Linear(new_input_features, out_features)

    # Initialize weights (can copy/interpolate from original if desired)
    nn.init.xavier_uniform_(new_first_linear.weight)
    if new_first_linear.bias is not None:
        nn.init.zeros_(new_first_linear.bias)

    # Replace in model
    model.classifier[0] = new_first_linear.to(DEVICE)

    print(f"  Updated classifier input: {first_linear.in_features} → {new_input_features}")

In [ ]:
#
# Update next layer's input channels after current layer is pruned
#
def update_next_layer_input_channels(model, next_layer_name, selected_channels):
    """
    Update the next layer's input channels to match the current layer's pruned output channels.
    This must be done IMMEDIATELY after pruning to maintain channel consistency.

    Args:
        model: VGG16-BN model
        next_layer_name: Name of next conv layer (e.g., 'features.10')
        selected_channels: Indices of output channels kept in current layer
    """
    # Get the next conv layer from the model
    parts = next_layer_name.split('.')
    module_name = parts[0]  # 'features'
    layer_idx = int(parts[1])

    sequential_module = getattr(model, module_name)
    next_conv = sequential_module[layer_idx]

    if not isinstance(next_conv, nn.Conv2d):
        return  # Not a conv layer, skip

    # Update input channels by slicing weights
    old_weight = next_conv.weight.data.cpu()  # (n_out, c_in, kh, kw)
    new_weight = old_weight[:, selected_channels, :, :]  # (n_out, c_pruned, kh, kw)

    # Create new conv layer with updated input channels
    new_conv = nn.Conv2d(
        in_channels=len(selected_channels),
        out_channels=next_conv.out_channels,
        kernel_size=next_conv.kernel_size,
        stride=next_conv.stride,
        padding=next_conv.padding,
        dilation=next_conv.dilation,
        groups=1,
        bias=next_conv.bias is not None
    )

    # Copy weights
    new_conv.weight.data = new_weight.to(DEVICE)
    if next_conv.bias is not None:
        new_conv.bias.data = next_conv.bias.data.to(DEVICE)

    # Replace the layer
    sequential_module[layer_idx] = new_conv

    print(f"  Updated {next_layer_name}: {old_weight.shape[1]} → {len(selected_channels)} input channels")

#
# Update classifier input dimension after pruning last conv layer
#
def update_classifier_input_dimension(model, last_conv_channels):
    """
    Update the first linear layer's input dimension after pruning last conv block.

    When the last conv layer (features.40) is pruned, the flattened feature size changes.

    Args:
        model: VGG16-BN model
        last_conv_channels: Number of output channels in last conv layer after pruning
    """
    # Calculate new flattened size (for CIFAR: 1x1 spatial after adaptive pooling)
    new_input_features = last_conv_channels * 1 * 1  # VGG uses AdaptiveAvgPool2d((1,1))

    # Get current first linear layer
    first_linear = model.classifier[0]
    out_features = first_linear.out_features

    # Create new linear layer with correct input dimension
    new_first_linear = nn.Linear(new_input_features, out_features)

    # Initialize weights (can copy/interpolate from original if desired)
    nn.init.xavier_uniform_(new_first_linear.weight)
    if new_first_linear.bias is not None:
        nn.init.zeros_(new_first_linear.bias)

    # Replace in model
    model.classifier[0] = new_first_linear.to(DEVICE)

    print(f"  Updated classifier input: {first_linear.in_features} → {new_input_features}")

## Part 7: Sequential Layer Pruning Pipeline

### Implementation Note: He et al. 2017 Method

The He et al. 2017 method prunes **INPUT channels** of each layer:

1. **For each layer L (starting from features.7)**:
   - Use LASSO to select which INPUT channels to keep
   - Reconstruct weights for layer L with fewer input channels
   - Update the PREVIOUS layer (L-1) to have matching output channels

2. **Special case for features.7 (first pruned layer)**:
   - Prunes input channels coming from features.3
   - Updates features.3's output channels to match

3. **Channel dependency management**:
   - When layer L prunes its input channels to k channels
   - Layer L-1 must have exactly k output channels
   - This cascades through the network maintaining consistency

This is why the sparsity is applied to INPUT channels, not output channels.

In [ ]:
#
# Prune output channels for first layer (magnitude-based)
#
def prune_output_channels_magnitude(model, layer_name, target_channels):
    """
    Prune output channels using magnitude-based selection.
    Used for the first layer (features.7) where we can't modify the previous layer.

    Args:
        model: VGG16-BN model
        layer_name: Name of layer to prune
        target_channels: Number of output channels to keep

    Returns:
        selected_channels: Indices of selected output channels
        W_new: New weights with selected output channels
    """
    # Get the conv layer
    conv_layer = None
    for name, module in model.named_modules():
        if name == layer_name and isinstance(module, nn.Conv2d):
            conv_layer = module
            break

    if conv_layer is None:
        raise ValueError(f"Layer {layer_name} not found")

    # Get weights
    W = conv_layer.weight.data.cpu()  # (n_out, c_in, kh, kw)
    n_out, c_in, kh, kw = W.shape

    # Compute L2 norm of each filter (output channel)
    filter_norms = torch.norm(W.view(n_out, -1), p=2, dim=1)  # (n_out,)

    # Select top-k filters by magnitude
    _, selected_channels = torch.topk(filter_norms, target_channels, largest=True)
    selected_channels = sorted(selected_channels.numpy())

    # Extract selected filters
    W_new = W[selected_channels, :, :, :]  # (target_channels, c_in, kh, kw)

    return np.array(selected_channels), W_new


### Note: This function is deprecated and kept only for reference
The main pipeline now uses LASSO regression for all layers as required by He et al. 2017.

In [ ]:
#
# Main sequential pruning function with LASSO and weight reconstruction
#
def prune_model_sequentially(model, layer_sparsity_dict, calibration_loader, test_loader, dataset_name='cifar10'):
    """
    Apply structured pruning layer by layer using He et al. 2017 method.

    Uses LASSO regression for channel selection and weight reconstruction
    via least squares to maintain network functionality.
    """
    pruned_model = copy.deepcopy(model)
    pruning_log = {}

    # Get all conv layers
    conv_layers = []
    for name, module in pruned_model.named_modules():
        if isinstance(module, nn.Conv2d) and 'features' in name:
            conv_layers.append((name, module))

    # Start from features.7
    start_idx = None
    for idx, (name, _) in enumerate(conv_layers):
        if name == 'features.7':
            start_idx = idx
            break

    print("\n" + "="*80)
    print(f"SEQUENTIAL CHANNEL PRUNING (He et al. 2017) - {dataset_name.upper()}")
    print("="*80)
    print(f"Starting from layer: {conv_layers[start_idx][0]}")
    print(f"Total layers to prune: {len(conv_layers) - start_idx}")
    print("="*80)

    for layer_idx in range(start_idx, len(conv_layers)):
        layer_name = conv_layers[layer_idx][0]
        sparsity = layer_sparsity_dict.get(layer_name, 0.0)

        # Get fresh reference to the layer (important after updates!)
        conv_layer = None
        for name, module in pruned_model.named_modules():
            if name == layer_name and isinstance(module, nn.Conv2d):
                conv_layer = module
                break

        if sparsity == 0.0:
            print(f"\n[{layer_idx - start_idx + 1}/{len(conv_layers) - start_idx}] {layer_name}: Skipping (sparsity=0.0)")
            continue

        print(f"\n[{layer_idx - start_idx + 1}/{len(conv_layers) - start_idx}] Processing: {layer_name}")
        print("-" * 80)

        current_out_channels = conv_layer.out_channels
        current_in_channels = conv_layer.in_channels
        target_in_channels = int(current_in_channels * (1 - sparsity))

        print(f"  Current: {current_in_channels} in → {current_out_channels} out")
        print(f"  Target sparsity: {sparsity:.1%}")
        print(f"  Target in channels: {target_in_channels}")

        # Step 1: Collect input-output pairs
        print(f"  Step 1: Collecting input-output pairs...")
        X_unf, Y, W_mat, conv_params = collect_layer_io_pairs(
            pruned_model, layer_name, calibration_loader
        )

        # Step 2: LASSO channel selection (select INPUT channels)
        print(f"  Step 2: LASSO regression for channel selection...")
        selected_in_channels, beta_values = solve_channel_selection_lasso(
            X_unf, Y, W_mat, target_in_channels, conv_params
        )
        print(f"  Selected {len(selected_in_channels)} input channels via LASSO")

        # Step 3: Weight reconstruction via least squares
        print(f"  Step 3: Weight reconstruction via least squares...")
        W_new = reconstruct_weights(
            X_unf, Y, selected_in_channels, beta_values, conv_params
        )

        # Update current layer's architecture with reconstructed weights
        print(f"  Updating {layer_name} architecture with reconstructed weights...")
        # Keep all output channels but with reduced input channels
        all_output_channels = np.arange(current_out_channels)
        update_layer_architecture(pruned_model, layer_name, all_output_channels, W_new)

        # Update PREVIOUS layer to match selected input channels
        # The previous layer's output channels must match our selected input channels
        if layer_idx > start_idx:
            prev_layer_name = conv_layers[layer_idx - 1][0]
            print(f"  Updating previous layer {prev_layer_name} to match selected channels...")

            # Get previous layer
            prev_conv_layer = None
            for name, module in pruned_model.named_modules():
                if name == prev_layer_name and isinstance(module, nn.Conv2d):
                    prev_conv_layer = module
                    break

            if prev_conv_layer is not None:
                # Extract only the selected output channels from previous layer
                prev_weights = prev_conv_layer.weight.data[selected_in_channels, :, :, :]

                # Update previous layer architecture (this also handles BatchNorm)
                update_layer_architecture(pruned_model, prev_layer_name, selected_in_channels, prev_weights.cpu())

        elif layer_idx == start_idx:
            # For the first layer (features.7), update features.3 output channels
            # features.3 is the last layer of the first block
            prev_layer_name = 'features.3'
            print(f"  Updating first block output layer {prev_layer_name}...")

            # Get features.3 layer
            prev_conv_layer = None
            for name, module in pruned_model.named_modules():
                if name == prev_layer_name and isinstance(module, nn.Conv2d):
                    prev_conv_layer = module
                    break

            if prev_conv_layer is not None:
                # Extract only the selected output channels
                prev_weights = prev_conv_layer.weight.data[selected_in_channels, :, :, :]
                update_layer_architecture(pruned_model, prev_layer_name, selected_in_channels, prev_weights.cpu())

        # Special handling for last layer
        if layer_idx == len(conv_layers) - 1:
            # This is the last conv layer - update classifier input dimension
            last_conv = None
            for name, module in pruned_model.named_modules():
                if name == layer_name and isinstance(module, nn.Conv2d):
                    last_conv = module
                    break
            if last_conv:
                print(f"  Updating classifier input dimension...")
                update_classifier_input_dimension(pruned_model, last_conv.out_channels)

        # Evaluate
        pruned_model.eval()
        with torch.no_grad():
            top1, top5 = evaluate_accuracy(pruned_model, test_loader)
        print(f"  Accuracy: Top-1={top1:.2f}%, Top-5={top5:.2f}%")

        pruning_log[layer_name] = {
            'original_in_channels': current_in_channels,
            'pruned_in_channels': len(selected_in_channels),
            'original_out_channels': current_out_channels,
            'sparsity': sparsity,
            'accuracy_after': top1,
            'method': 'LASSO + Weight Reconstruction'
        }

        # Save checkpoint
        checkpoint_dir = os.path.join(BASE_PATH, 'task2', 'checkpoints', f'task2_{dataset_name}')
        os.makedirs(checkpoint_dir, exist_ok=True)
        torch.save({
            'model_state': pruned_model.state_dict(),
            'accuracy': top1
        }, os.path.join(checkpoint_dir, f'layer_{layer_name.replace(".", "_")}.pt'))

    print("\n" + "="*80)
    print("SEQUENTIAL PRUNING COMPLETE")
    print("="*80)

    return pruned_model, pruning_log

## Part 8: Final Fine-tuning (ONLY After ALL Pruning)

In [ ]:
#
# Final fine-tuning after all pruning complete
#
def final_finetuning(model, train_loader, test_loader, checkpoint_dir, max_epochs=100, patience=25):
    """
    Fine-tune ONLY after ALL layers have been pruned.

    This is the ONLY training that happens in the entire task!
    Everything before this is mathematical computation.

    Settings:
    - Loss: CrossEntropyLoss (as required)
    - Epochs: max_epochs (default 100)
    - Early stopping: patience epochs
    - Learning rate: 0.01 (standard VGG rate)
    - Optimizer: SGD with momentum
    """
    print("\n" + "="*80)
    print("FINAL FINE-TUNING (After ALL pruning complete)")
    print("="*80)

    # Check for existing checkpoint
    final_checkpoint = os.path.join(checkpoint_dir, 'final_finetuning.pt')
    start_epoch = 0
    best_accuracy = 0.0

    if os.path.exists(final_checkpoint):
        print(f"Loading checkpoint from {final_checkpoint}")
        checkpoint = torch.load(final_checkpoint, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state'])
        start_epoch = checkpoint['epoch']
        best_accuracy = checkpoint.get('best_accuracy', 0.0)
        print(f"Resuming from epoch {start_epoch}, best accuracy: {best_accuracy:.2f}%")

    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)
    criterion = nn.CrossEntropyLoss()

    epochs_without_improvement = 0

    for epoch in range(start_epoch, max_epochs):
        # Training
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({'loss': train_loss/(pbar.n+1), 'acc': 100.*correct/total})

        train_acc = 100. * correct / total

        # Validation
        model.eval()
        with torch.no_grad():
            test_top1, test_top5 = evaluate_accuracy(model, test_loader)

        scheduler.step()

        print(f"Epoch {epoch+1}: Train Loss={train_loss/len(train_loader):.4f}, "
              f"Train Acc={train_acc:.2f}%, Test Top-1={test_top1:.2f}%, Test Top-5={test_top5:.2f}%")

        # Save best model
        if test_top1 > best_accuracy:
            best_accuracy = test_top1
            epochs_without_improvement = 0
            torch.save({
                'model_state': model.state_dict(),
                'epoch': epoch + 1,
                'best_accuracy': best_accuracy
            }, os.path.join(checkpoint_dir, 'best_model.pt'))
            print(f"  ✓ New best accuracy: {best_accuracy:.2f}%")
        else:
            epochs_without_improvement += 1

        # Save checkpoint
        torch.save({
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'epoch': epoch + 1,
            'accuracy': test_top1,
            'best_accuracy': best_accuracy
        }, final_checkpoint)

        # Early stopping
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping after {epoch+1} epochs (no improvement for {patience} epochs)")
            break

    # Load best model
    best_checkpoint = torch.load(os.path.join(checkpoint_dir, 'best_model.pt'), map_location=DEVICE)
    model.load_state_dict(best_checkpoint['model_state'])

    print("\n" + "="*80)
    print(f"FINE-TUNING COMPLETE - Best Accuracy: {best_accuracy:.2f}%")
    print("="*80)

    return model

## Part 9: Complete Pipeline for CIFAR-10

In [ ]:
#
# Complete structured pruning pipeline for CIFAR-10
#
def run_structured_pruning_cifar10():
    """
    Complete structured pruning pipeline for CIFAR-10.

    Uses He et al. 2017 regression-based method:
    - LASSO for channel selection
    - Weight reconstruction via least squares
    - Final fine-tuning after all pruning
    """
    print("="*80)
    print("TASK 2: STRUCTURED CHANNEL PRUNING - CIFAR-10")
    print("Method: He et al. 2017 (LASSO + Weight Reconstruction)")
    print("="*80)

    # Step 1: Load model and data
    print("\n[Step 1] Loading pretrained model and dataset...")
    model = load_pretrained_model('cifar10')
    train_dataset, test_dataset = load_dataset('cifar10')

    # Step 2: Create data loaders
    print("[Step 2] Creating data loaders...")
    calibration_indices = list(range(CALIBRATION_SAMPLES))
    calibration_subset = Subset(train_dataset, calibration_indices)
    calibration_loader = DataLoader(calibration_subset, batch_size=BATCHSIZE, shuffle=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCHSIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCHSIZE, shuffle=False, num_workers=2)

    # Baseline evaluation
    print("[Baseline] Evaluating original model...")
    baseline_top1, baseline_top5 = evaluate_accuracy(model, test_loader)
    print(f"Baseline accuracy: Top-1={baseline_top1:.2f}%, Top-5={baseline_top5:.2f}%")

    # Step 3: Apply structured pruning using LASSO and weight reconstruction
    print("\n[Step 3] Applying structured pruning (LASSO + weight reconstruction)...")
    pruned_model, pruning_log = prune_model_sequentially(
        model, cifar10_layer_sparsity, calibration_loader, test_loader, 'cifar10'
    )

    # Show summary
    print("\n" + "="*80)
    print("PRUNING COMPLETE - Summary:")
    print("="*80)
    for layer_name, info in pruning_log.items():
        print(f"{layer_name}: {info['original_in_channels']} → {info['pruned_in_channels']} in channels "
              f"({info['sparsity']:.0%} sparsity), Acc={info['accuracy_after']:.2f}%, "
              f"Method={info['method']}")

    # Calculate overall sparsity
    total_original_params = sum(p.numel() for p in model.parameters())
    total_pruned_params = sum(p.numel() for p in pruned_model.parameters())
    overall_sparsity = 1 - (total_pruned_params / total_original_params)
    print(f"\nOverall parameter reduction: {overall_sparsity:.2%}")
    print(f"Original params: {total_original_params:,}")
    print(f"Pruned params: {total_pruned_params:,}")

    # Step 4: Final fine-tuning
    print("\n[Step 4] Final fine-tuning (ONLY training in entire pipeline)...")
    checkpoint_dir = os.path.join(BASE_PATH, 'task2', 'checkpoints', 'task2_cifar10')
    os.makedirs(checkpoint_dir, exist_ok=True)
    final_model = final_finetuning(pruned_model, train_loader, test_loader, checkpoint_dir)

    # Step 5: Profile and save
    print("\n[Step 5] Profiling final model...")
    final_top1, final_top5 = evaluate_accuracy(final_model, test_loader)
    model_size_mb = get_model_size_mb(final_model)
    profile_metrics = profile_model(final_model, test_loader)
    macs = count_model_macs(final_model)

    results = {
        'dataset': 'cifar10',
        'method': 'He et al. 2017 - LASSO + Weight Reconstruction',
        'baseline_top1': baseline_top1,
        'baseline_top5': baseline_top5,
        'final_top1': final_top1,
        'final_top5': final_top5,
        'accuracy_drop': baseline_top1 - final_top1,
        'model_size_mb': model_size_mb,
        'total_params': total_pruned_params,
        'original_params': total_original_params,
        'overall_sparsity': overall_sparsity,
        'latency_ms': profile_metrics['latency_ms'],
        'peak_memory_mb': profile_metrics['peak_memory_mb'],
        'macs': macs,
        'pruning_log': pruning_log
    }

    # Save results
    results_path = os.path.join(BASE_PATH, 'task2' 'results', 'task2_cifar10_results.json')
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)

    # Save model
    model_path = os.path.join(BASE_PATH, 'task2', 'models', 'task2_cifar10_pruned.pt')
    torch.save(final_model.state_dict(), model_path)

    print("\n" + "="*80)
    print("CIFAR-10 STRUCTURED PRUNING COMPLETE")
    print("="*80)
    print(f"Baseline accuracy: {baseline_top1:.2f}%")
    print(f"Final accuracy: {final_top1:.2f}%")
    print(f"Accuracy drop: {results['accuracy_drop']:.2f}%")
    print(f"Model size: {model_size_mb:.2f} MB")
    print(f"Overall sparsity: {overall_sparsity:.2%}")
    print(f"Method: {results['method']}")
    print(f"Results saved to: {results_path}")

    return final_model, results

## Part 10: Complete Pipeline for CIFAR-100

In [ ]:
#
# Complete structured pruning pipeline for CIFAR-100
#
def run_structured_pruning_cifar100():
    """
    Complete structured pruning pipeline for CIFAR-100.

    Uses He et al. 2017 regression-based method:
    - LASSO for channel selection
    - Weight reconstruction via least squares
    - Final fine-tuning after all pruning
    """
    print("="*80)
    print("TASK 2: STRUCTURED CHANNEL PRUNING - CIFAR-100")
    print("Method: He et al. 2017 (LASSO + Weight Reconstruction)")
    print("="*80)

    # Step 1: Load model and data
    print("\n[Step 1] Loading pretrained model and dataset...")
    model = load_pretrained_model('cifar100')
    train_dataset, test_dataset = load_dataset('cifar100')

    # Step 2: Create data loaders
    print("[Step 2] Creating data loaders...")
    calibration_indices = list(range(CALIBRATION_SAMPLES))
    calibration_subset = Subset(train_dataset, calibration_indices)
    calibration_loader = DataLoader(calibration_subset, batch_size=BATCHSIZE, shuffle=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCHSIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCHSIZE, shuffle=False, num_workers=2)

    # Baseline evaluation
    print("[Baseline] Evaluating original model...")
    baseline_top1, baseline_top5 = evaluate_accuracy(model, test_loader)
    print(f"Baseline accuracy: Top-1={baseline_top1:.2f}%, Top-5={baseline_top5:.2f}%")

    # Step 3: Apply structured pruning using LASSO and weight reconstruction
    print("\n[Step 3] Applying structured pruning (LASSO + weight reconstruction)...")
    pruned_model, pruning_log = prune_model_sequentially(
        model, cifar100_layer_sparsity, calibration_loader, test_loader, 'cifar100'
    )

    # Show summary
    print("\n" + "="*80)
    print("PRUNING COMPLETE - Summary:")
    print("="*80)
    for layer_name, info in pruning_log.items():
        print(f"{layer_name}: {info['original_in_channels']} → {info['pruned_in_channels']} in channels "
              f"({info['sparsity']:.0%} sparsity), Acc={info['accuracy_after']:.2f}%, "
              f"Method={info['method']}")

    # Calculate overall sparsity
    total_original_params = sum(p.numel() for p in model.parameters())
    total_pruned_params = sum(p.numel() for p in pruned_model.parameters())
    overall_sparsity = 1 - (total_pruned_params / total_original_params)
    print(f"\nOverall parameter reduction: {overall_sparsity:.2%}")
    print(f"Original params: {total_original_params:,}")
    print(f"Pruned params: {total_pruned_params:,}")

    # Step 4: Final fine-tuning
    print("\n[Step 4] Final fine-tuning (ONLY training in entire pipeline)...")
    checkpoint_dir = os.path.join(BASE_PATH, 'task2', 'checkpoints', 'task2_cifar100')
    os.makedirs(checkpoint_dir, exist_ok=True)
    final_model = final_finetuning(pruned_model, train_loader, test_loader, checkpoint_dir)

    # Step 5: Profile and save
    print("\n[Step 5] Profiling final model...")
    final_top1, final_top5 = evaluate_accuracy(final_model, test_loader)
    model_size_mb = get_model_size_mb(final_model)
    profile_metrics = profile_model(final_model, test_loader)
    macs = count_model_macs(final_model)

    results = {
        'dataset': 'cifar100',
        'method': 'He et al. 2017 - LASSO + Weight Reconstruction',
        'baseline_top1': baseline_top1,
        'baseline_top5': baseline_top5,
        'final_top1': final_top1,
        'final_top5': final_top5,
        'accuracy_drop': baseline_top1 - final_top1,
        'model_size_mb': model_size_mb,
        'total_params': total_pruned_params,
        'original_params': total_original_params,
        'overall_sparsity': overall_sparsity,
        'latency_ms': profile_metrics['latency_ms'],
        'peak_memory_mb': profile_metrics['peak_memory_mb'],
        'macs': macs,
        'pruning_log': pruning_log
    }

    # Save results
    results_path = os.path.join(BASE_PATH, 'task2', 'results', 'task2_cifar100_results.json')
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)

    # Save model
    model_path = os.path.join(BASE_PATH, 'task2', 'models', 'task2_cifar100_pruned.pt')
    torch.save(final_model.state_dict(), model_path)

    print("\n" + "="*80)
    print("CIFAR-100 STRUCTURED PRUNING COMPLETE")
    print("="*80)
    print(f"Baseline accuracy: {baseline_top1:.2f}%")
    print(f"Final accuracy: {final_top1:.2f}%")
    print(f"Accuracy drop: {results['accuracy_drop']:.2f}%")
    print(f"Model size: {model_size_mb:.2f} MB")
    print(f"Overall sparsity: {overall_sparsity:.2%}")
    print(f"Method: {results['method']}")
    print(f"Results saved to: {results_path}")

    return final_model, results

## Part 11: Execute Pipeline

Run the complete structured pruning pipeline for both datasets.

## ⚠️ Important: Re-run Required

**The implementation has been fixed to properly use the He et al. 2017 method.**

Previous execution used magnitude-based pruning instead of LASSO regression. The notebook now correctly implements:

1. **LASSO Channel Selection**: Uses regression to identify important channels
2. **Weight Reconstruction**: Applies least squares to maintain network functionality
3. **No Intermediate Training**: Only mathematical computation during pruning

**To get correct results:**
1. Clear all outputs (Kernel → Restart & Clear Output)
2. Run all cells from the beginning
3. The execution in cells 36-37 will now use the proper He et al. 2017 algorithm

**Expected differences:**
- LASSO may select different channels than magnitude-based pruning
- Weight reconstruction should better preserve network functionality
- May achieve different accuracy/sparsity trade-offs
- Pruning phase may take slightly longer due to LASSO regression

In [24]:
# Run CIFAR-10 structured pruning
cifar10_model, cifar10_results = run_structured_pruning_cifar10()

TASK 2: STRUCTURED CHANNEL PRUNING - CIFAR-10
Method: He et al. 2017 (LASSO + Weight Reconstruction)

[Step 1] Loading pretrained model and dataset...
Loaded pretrained model from /content/drive/MyDrive/AI624-Edge-Devices/pa1.1/models/cifar10_vgg16.pt
  Classifier input features: 512
  Number of classes: 10
[Step 2] Creating data loaders...
[Baseline] Evaluating original model...
Baseline accuracy: Top-1=94.16%, Top-5=99.71%

[Step 3] Applying structured pruning (LASSO + weight reconstruction)...

SEQUENTIAL CHANNEL PRUNING (He et al. 2017) - CIFAR10
Starting from layer: features.7
Total layers to prune: 11

[1/11] Processing: features.7
--------------------------------------------------------------------------------
  Current: 64 in → 128 out
  Target sparsity: 70.0%
  Target in channels: 19
  Step 1: Collecting input-output pairs...
  Step 2: LASSO regression for channel selection...
  Selected 19 input channels via LASSO
  Step 3: Weight reconstruction via least squares...
  Updatin

In [25]:
# Run CIFAR-100 structured pruning
cifar100_model, cifar100_results = run_structured_pruning_cifar100()

TASK 2: STRUCTURED CHANNEL PRUNING - CIFAR-100
Method: He et al. 2017 (LASSO + Weight Reconstruction)

[Step 1] Loading pretrained model and dataset...
Loaded pretrained model from /content/drive/MyDrive/AI624-Edge-Devices/pa1.1/models/cifar100_vgg16.pt
  Classifier input features: 512
  Number of classes: 100
[Step 2] Creating data loaders...
[Baseline] Evaluating original model...
Baseline accuracy: Top-1=71.34%, Top-5=88.94%

[Step 3] Applying structured pruning (LASSO + weight reconstruction)...

SEQUENTIAL CHANNEL PRUNING (He et al. 2017) - CIFAR100
Starting from layer: features.7
Total layers to prune: 11

[1/11] Processing: features.7
--------------------------------------------------------------------------------
  Current: 64 in → 128 out
  Target sparsity: 60.0%
  Target in channels: 25
  Step 1: Collecting input-output pairs...
  Step 2: LASSO regression for channel selection...
  Selected 25 input channels via LASSO
  Step 3: Weight reconstruction via least squares...
  Upd

Epoch 1/100: 100%|██████████| 391/391 [00:09<00:00, 40.26it/s, loss=4.16, acc=4.91]


Epoch 1: Train Loss=4.1526, Train Acc=4.91%, Test Top-1=11.88%, Test Top-5=39.44%
  ✓ New best accuracy: 11.88%


Epoch 2/100: 100%|██████████| 391/391 [00:09<00:00, 39.92it/s, loss=3.34, acc=14]


Epoch 2: Train Loss=3.3082, Train Acc=14.02%, Test Top-1=20.91%, Test Top-5=55.15%
  ✓ New best accuracy: 20.91%


...

Epoch 98: Train Loss=0.0081, Train Acc=99.87%, Test Top-1=54.47%, Test Top-5=79.27%


Epoch 99/100: 100%|██████████| 391/391 [00:09<00:00, 40.38it/s, loss=0.00818, acc=99.9]


Epoch 99: Train Loss=0.0081, Train Acc=99.88%, Test Top-1=54.36%, Test Top-5=79.27%


Epoch 100/100: 100%|██████████| 391/391 [00:09<00:00, 40.36it/s, loss=0.00762, acc=99.9]


Epoch 100: Train Loss=0.0075, Train Acc=99.88%, Test Top-1=54.21%, Test Top-5=79.34%

Early stopping after 100 epochs (no improvement for 25 epochs)

FINE-TUNING COMPLETE - Best Accuracy: 54.99%

[Step 5] Profiling final model...

CIFAR-100 STRUCTURED PRUNING COMPLETE
Baseline accuracy: 71.34%
Final accuracy: 54.99%
Accuracy drop: 16.35%
Model size: 5.21 MB
Overall sparsity: 91.14%
Method: He et al. 2017 - LASSO + Weight Reconstruction
Results saved to: /content/drive/MyDrive/AI624-Edge-Devices/pa1.1/task2/results/task2_cifar100_results.json
